# Kedro Data Catalog — Demo Notebook

This notebook demonstrates how to interact with the datasets produced by the `hdf_pipelines` Kedro project,
using the `DataCatalog` API as the single access point for all pipeline outputs.

**How to launch** (from the `pipelines/` directory):
```bash
kedro jupyter notebook
```
This automatically injects a `catalog` variable into the kernel. The bootstrap cell below does the same thing
manually, so the notebook also works when opened directly from Jupyter or VS Code.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project

# # Path.cwd().parent resolves to pipelines/ when launched via `kedro jupyter notebook`
project_root = Path.cwd().parent
bootstrap_project(project_root)

session = KedroSession.create(project_path=project_root)
catalog = session.load_context().catalog

# print(f"Project root : {project_root}")
# print("Kedro session ready.")

## 1. Available Datasets

Every dataset registered in `conf/base/catalog.yml` is accessible through `catalog.load("<name>")`.
The cell below lists all registered names, grouped by their data layer prefix.

In [ ]:
all_datasets = catalog
print(f"Total registered datasets: {len(all_datasets)}\n")
for name in all_datasets:
    print(f"  {name}")

## 2. Layer `02_intermediate` — Cleaned Data

These are the outputs of the **data ingestion and cleaning** pipeline. Raw CSVs have been parsed,
typed, and standardized into a consistent daily-grain format ready for aggregation.

In [ ]:
demand_cleaned = catalog.load("demand_cleaned")

print(f"Shape      : {demand_cleaned.shape}")
print(
    f"Date range : {demand_cleaned['date'].min().date()} → {demand_cleaned['date'].max().date()}"
)
print(f"Columns    : {demand_cleaned.columns.tolist()}\n")
demand_cleaned.head()

In [ ]:
exogenous_cleaned = catalog.load("exogenous_cleaned")

print(f"Shape   : {exogenous_cleaned.shape}")
print(f"Columns : {exogenous_cleaned.columns.tolist()}\n")
exogenous_cleaned.head()

## 3. Layer `03_primary` — Temporal Hierarchy

The cleaned demand signal is aggregated into three temporal granularities.
Each is its own catalog entry and Parquet file — no re-processing needed to switch between them.

| Dataset | Granularity | Priority |
|---|---|---|
| `demand_monthly` | Monthly | **Primary** — main modeling target |
| `demand_weekly`  | Weekly  | Secondary — operational complement |
| `demand_daily`   | Daily   | Exploratory only |

In [ ]:
demand_monthly = catalog.load("demand_monthly")
demand_weekly = catalog.load("demand_weekly")
demand_daily = catalog.load("demand_daily")

for label, df in [
    ("monthly", demand_monthly),
    ("weekly", demand_weekly),
    ("daily", demand_daily),
]:
    print(f"{label:>8} → {df.shape[0]:>4} rows  |  columns: {df.columns.tolist()}")

print()
demand_monthly.head()

In [ ]:
exogenous_monthly = catalog.load("exogenous_monthly")

print(f"Shape   : {exogenous_monthly.shape}")
print(f"Columns : {exogenous_monthly.columns.tolist()}\n")
exogenous_monthly.head()

## 4. Visualization — Demand Across Temporal Layers

The three panels below show the same underlying demand signal at different resolutions.
Monthly smooths out short-term noise; daily reveals intra-month patterns.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=False)

layers = [
    (
        demand_monthly,
        "month_start_date",
        "monthly_demand",
        "Monthly demand",
        "steelblue",
    ),
    (demand_weekly, "week_start_date", "weekly_demand", "Weekly demand", "darkorange"),
    (demand_daily, "date", "daily_demand", "Daily demand", "seagreen"),
]

for ax, (df, date_col, demand_col, title, color) in zip(axes, layers):
    ax.plot(df[date_col], df[demand_col], color=color, linewidth=1.3)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_ylabel("Units")
    ax.grid(alpha=0.3)

plt.suptitle("Temporal hierarchy — same SKU, three granularities", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Exogenous Variables Over Time

The three exogenous variables (`pfizer_limited`, `surgifoam_limited`, `rebate_target`) are
binary/continuous signals aligned to the monthly demand layer. They will be used as
model regressors in the CatBoost and SARIMAX pipelines.

In [ ]:
exog_cols = ["pfizer_limited", "surgifoam_limited", "rebate_target"]
colors = ["crimson", "mediumpurple", "goldenrod"]

fig, axes = plt.subplots(len(exog_cols), 1, figsize=(13, 7), sharex=True)

for ax, col, color in zip(axes, exog_cols, colors):
    ax.step(
        exogenous_monthly["month_start_date"],
        exogenous_monthly[col],
        where="mid",
        color=color,
        linewidth=1.4,
    )
    ax.set_title(col, fontsize=10, fontweight="bold")
    ax.grid(alpha=0.3)

plt.suptitle("Exogenous variables — monthly granularity", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()